In [ ]:
import scanpy as sc

# ==========================================
# 1. 读取全量数据
# ==========================================
adata = sc.read_h5ad('adata_snRNA_pig_processed.h5ad')
print("全量数据概况：\n", adata)

In [ ]:
sc.set_figure_params(dpi=150, figsize=(5, 5))

# 1. 按照【细胞类型】查看 UMAP
sc.pl.umap(adata, color='cell_type', title='UMAP by Cell Type')

# 2. 按照【聚类编号】查看 UMAP
sc.pl.umap(adata, color='leiden', title='UMAP by Leiden Clusters')

# 3. 按照【时间点】或【样本】查看批次效应分布
sc.pl.umap(adata, color=['time_point', 'sample_id'], wspace=0.3)

In [ ]:
# ==========================================
# 1. 提取指定的这三类内皮细胞
# ==========================================
ec_types = [
    'Arterial VEC', 
    'Glomerular capillary EC', 
    'VEC'
]

# 提取子集并断开链接（.copy() 是不报错的关键）
adata_ec = adata[adata.obs['cell_type'].isin(ec_types)].copy()
print(f"✅ 成功提取！当前包含细胞数: {adata_ec.n_obs}, 基因数: {adata_ec.n_vars}")

In [ ]:

# ==========================================
# 2. 预处理与降维聚类
# ==========================================
# 过滤掉在这群内皮细胞中完全不表达的“死基因”
sc.pp.filter_genes(adata_ec, min_cells=3)

# 重新寻找仅针对内皮细胞的高变基因
sc.pp.highly_variable_genes(adata_ec, n_top_genes=2000, flavor='seurat')

# 创建高变基因的临时子集用于跑降维（让运算更快、更准）
adata_ec_hvg = adata_ec[:, adata_ec.var.highly_variable].copy()

# 核心降维聚类步骤
sc.pp.scale(adata_ec_hvg, max_value=10)
sc.tl.pca(adata_ec_hvg, svd_solver='arpack')
sc.pp.neighbors(adata_ec_hvg, n_neighbors=10, n_pcs=15) # n_pcs可根据需要调整
sc.tl.umap(adata_ec_hvg)
sc.tl.leiden(adata_ec_hvg, resolution=0.5, key_added='leiden_ec') # resolution控制分群数量

In [ ]:
# ==========================================
# 3. 将结果写回并可视化
# ==========================================
# 把算好的坐标和分类同步回包含全基因的内皮对象中
adata_ec.obsm['X_pca'] = adata_ec_hvg.obsm['X_pca']
adata_ec.obsm['X_umap'] = adata_ec_hvg.obsm['X_umap']
adata_ec.obs['leiden_ec'] = adata_ec_hvg.obs['leiden_ec']

# 画出最新的 UMAP 图
sc.set_figure_params(dpi=150, figsize=(5, 5))
sc.pl.umap(
    adata_ec, 
    color=['cell_type', 'leiden_ec'], 
    wspace=0.4, 
    title=['Original EC Annotations', 'New EC Leiden Clusters']
)

In [ ]:
# 增加 n_neighbors 让大群更聚拢，减少碎片化
sc.pp.neighbors(adata_ec_hvg, n_neighbors=10, n_pcs=15) 

# 重新跑 UMAP
sc.tl.umap(adata_ec_hvg) # min_dist调大一点能让点散开一点，方便看清结构

# 降低分辨率，试试 0.25 (你可以根据结果在 0.1 - 0.4 之间微调)
sc.tl.leiden(adata_ec_hvg, resolution=0.4, key_added='leiden_ec_opt')

# 覆盖回原数据
adata_ec.obsm['X_umap'] = adata_ec_hvg.obsm['X_umap']
adata_ec.obs['leiden_ec_opt'] = adata_ec_hvg.obs['leiden_ec_opt']

# 画图对比
sc.pl.umap(adata_ec, color=['cell_type', 'leiden_ec_opt'], wspace=0.4)


In [ ]:
sc.pl.umap(adata_ec, color=['leiden_ec_opt', 'time_point'], wspace=0.4, ncols=3)


In [ ]:
import matplotlib.pyplot as plt

# 画图，但加上 show=False 告诉它先别急着展示，等我存完再展示
sc.pl.umap(adata_ec, color=['cell_type', 'leiden_ec_opt','time_point'], wspace=0.6, show=False)

# 自定义保存的完整文件名和路径，bbox_inches='tight' 可以去除多余的白边
plt.savefig('EC_UMAP_Optimized_Clusters.pdf', dpi=300, bbox_inches='tight')
plt.savefig('EC_UMAP_Optimized_Clusters.png', dpi=300, bbox_inches='tight')

# 存完之后展示出来
plt.show()

In [ ]:
# 1. 寻找每个群的差异表达基因（Marker genes）
# 使用 wilcoxon 检验，对比每个群和其余所有群
sc.tl.rank_genes_groups(adata_ec, groupby='leiden_ec_opt', method='wilcoxon')

# 2. 可视化：画一个气泡图 (Dotplot)，查看每个群排名前 5 的 Marker 基因
sc.set_figure_params(dpi=150, figsize=(6, 4))
sc.pl.rank_genes_groups_dotplot(
    adata_ec, 
    n_genes=5,                  # 每个群展示 top5 基因
    groupby='leiden_ec_opt', 
    standard_scale='var',       # 对基因表达量进行 0-1 缩放，图更好看
    cmap='Blues'
)

In [ ]:

# 3. (可选) 如果你想把这些基因导出来成 Excel 表格发给生物学背景的同事看：
import pandas as pd
result = adata_ec.uns['rank_genes_groups']
groups = result['names'].dtype.names
marker_df = pd.DataFrame(
    {group + '_' + key: result[key][group]
    for group in groups for key in ['names', 'logfoldchanges', 'pvals_adj']}
)
# 保存到本地 CSV 文件
marker_df.to_csv("EC_subclusters_markers.csv", index=False)
print("Marker 基因表格已保存为 EC_subclusters_markers.csv！")


In [ ]:
import matplotlib.pyplot as plt

sc.set_figure_params(dpi=150, figsize=(8, 4)) # 稍微调宽一点，防止基因名挤在一起

# 画气泡图，关闭 dendrogram
sc.pl.rank_genes_groups_dotplot(
    adata_ec, 
    n_genes=5,                  
    groupby='leiden_ec_opt', 
    standard_scale='var',       
    cmap='Blues',
    dendrogram=False,           # 🔑 关键参数：关闭树状图，强制按数字顺序排列
    show=False
)

# 保存为 PDF 或 PNG
plt.savefig('EC_Dotplot_Ordered.pdf', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# 肾脏/血管内皮经典的“教科书级别” Marker 基因字典
classic_markers = {
    'Pan-EC (泛内皮)': ['PECAM1', 'CDH5', 'KDR'],         # 所有内皮都该有的
    'Arterial (动脉)': ['GJA5', 'HEY1', 'MECOM', 'EFNB2'], # 应该在群 2 亮起
    'Venous (静脉)': ['NR2F2', 'ACKR1', 'VWF'],            # 看看 0, 1, 3 谁亮
    'Capillary (毛细)': ['CA4', 'RGCC', 'PLVAP'],          # 看看 0, 1 谁亮
    'Glomerular (肾小球)': ['EHD3', 'PI16', 'PLAT'],       # 应该在群 4 亮起
    'Tip/Injury (新生/受损)': ['KDR', 'ESM1', 'FN1']       # 看看是不是群 6 亮起
}

# 确保基因名大小写匹配你的数据 (如果是小鼠把上面的字母改成首字母大写如 Pecam1)
sc.pl.dotplot(
    adata_ec, 
    classic_markers, 
    groupby='leiden_ec_opt', 
    standard_scale='var',
    dendrogram=False,
    cmap='Reds',      # 换个红色系看看
    figsize=(10, 4)
)


In [ ]:
# 创建一个字典，把数字映射成细胞类型名字
cluster2annotation = {
    '0': 'Capillary EC_1',
    '1': 'Capillary EC_2',
    '2': 'Arterial EC',
    '3': 'Venous EC',
    '4': 'Glomerular EC',
    '5': 'Low Quality',
    '6': 'Activated Arterial EC'
}

# 在 adata 中新建一列存入最终的名字
adata_ec.obs['Final_Cell_Type'] = adata_ec.obs['leiden_ec_opt'].map(cluster2annotation).astype('category')


In [ ]:
import matplotlib.pyplot as plt

# 换成纯英文的字典，解决方块乱码问题
classic_markers_en = {
    'Pan-Endothelial': ['PECAM1', 'CDH5', 'KDR'],         
    'Arterial': ['GJA5', 'HEY1', 'MECOM', 'EFNB2'], 
    'Venous': ['NR2F2', 'ACKR1', 'VWF'],            
    'Capillary': ['CA4', 'RGCC', 'PLVAP'],          
    'Glomerular': ['EHD3', 'PI16', 'PLAT'],       
    'Activated/Tip': ['ESM1', 'FN1']       
}

sc.set_figure_params(dpi=150, figsize=(10, 4))

# 用新名字 'Final_Cell_Type' 画图
sc.pl.dotplot(
    adata_ec, 
    classic_markers_en, 
    groupby='Final_Cell_Type',   # 使用我们刚命名的新列
    standard_scale='var',
    dendrogram=True,             # 开启聚类树，让相似的细胞自动排在一起
    cmap='Reds',      
    show=False
)

plt.savefig('Final_EC_Annotation_Dotplot.pdf', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
sc.pl.umap(
    adata_ec, 
    color=['Final_Cell_Type'], 
    legend_loc='right margin',
    title='Endothelial Cell Subclusters',
    show=False
)
plt.savefig('Final_EC_UMAP.pdf', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# 1. 剔除第 5 群 (Low Quality)
# 注意：最后加上 .copy() 是个好习惯，它会生成一个干净的新对象，防止后续报错
adata_ec_clean = adata_ec[adata_ec.obs['Final_Cell_Type'] != 'Low Quality'].copy()

# 为了图例好看，我们可以把不需要的 'Low Quality' 类别从分类体系中彻底清理掉
adata_ec_clean.obs['Final_Cell_Type'] = adata_ec_clean.obs['Final_Cell_Type'].cat.remove_unused_categories()

print(f"删除前细胞数: {adata_ec.n_obs}")
print(f"删除后细胞数: {adata_ec_clean.n_obs}")


# 2. 重新画一张干干净净的最终版 UMAP 图
sc.pl.umap(
    adata_ec_clean, 
    color=['Final_Cell_Type'], 
    legend_loc='right margin',
    title='Endothelial Cell Subclusters (Cleaned)',
    show=False
)
plt.savefig('Final_EC_UMAP_Cleaned.pdf', dpi=300, bbox_inches='tight')
plt.show()


# 3. 重新画一张干干净净的最终版气泡图
# 🔑 关键修复：针对清理后的数据，重新计算聚类树！
sc.tl.dendrogram(adata_ec_clean, groupby='Final_Cell_Type')

# 再次运行画图代码
import matplotlib.pyplot as plt
# 1. 手动定义一个完美的、符合逻辑的顺序
# 顺序：动脉(正常+激活) -> 毛细血管(1+2) -> 静脉 -> 肾小球
desired_order = [
    'Arterial EC',
    'Activated Arterial EC',
    'Venous EC',
    'Capillary EC_1',
    'Capillary EC_2',
    'Glomerular EC'
]

# 2. 将数据中的分类顺序强制修改为我们定义的顺序
adata_ec_clean.obs['Final_Cell_Type'] = adata_ec_clean.obs['Final_Cell_Type'].cat.reorder_categories(desired_order)

# 💎 优化后的“精装版” Marker 字典
optimized_markers = {
    'Pan-Endothelial': ['PECAM1', 'CDH5'],                  # 泛内皮保留这两个最稳的
    'Arterial': ['GJA5', 'MECOM', 'EFNB2'],                 # 删掉 HEY1，保留这三个又大又红的
    'Activated/Tip': ['CDH13', 'FN1'] ,                      # 删掉 ESM1，换成 CDH13
    'Venous': ['VWF'],                                      # 静脉内皮保留 VWF 即可，它表现得非常好
    'Capillary': ['ADGRF5', 'PTPRG', 'RGCC', 'PLVAP'],      # 加入了你数据里真实的强表达基因
    'Glomerular': ['EHD3', 'PLAT', 'GATA5']                 # 删掉 PI16，换成你数据里的 GATA5
}

sc.set_figure_params(dpi=150, figsize=(10, 4))

sc.pl.dotplot(
    adata_ec_clean, 
    optimized_markers, 
    groupby='Final_Cell_Type', 
    standard_scale='var',
    dendrogram=False,      
    cmap='Blues',      
    show=False
)
plt.savefig('Final_EC_Annotation_Dotplot_Optimized.pdf', dpi=300, bbox_inches='tight')
plt.show()



In [ ]:
# 1. 把聚类结果转换成字符串，将 '6' 替换为 '5'
adata_ec_clean.obs['leiden_ec_opt'] = adata_ec_clean.obs['leiden_ec_opt'].astype(str)
adata_ec_clean.obs['leiden_ec_opt'] = adata_ec_clean.obs['leiden_ec_opt'].replace('6', '5')

# 2. 重新转换回 category 类型，并按数字大小排序（保证图例是从 0 到 5 依次排列）
# 提取目前所有的唯一值并按数字排序
sorted_cats = sorted(adata_ec_clean.obs['leiden_ec_opt'].unique(), key=int)
adata_ec_clean.obs['leiden_ec_opt'] = adata_ec_clean.obs['leiden_ec_opt'].astype('category').cat.reorder_categories(sorted_cats)


In [ ]:
# 画图，但加上 show=False 告诉它先别急着展示，等我存完再展示
sc.pl.umap(adata_ec_clean, color=['cell_type', 'leiden_ec_opt','Final_Cell_Type'], wspace=0.6, show=False)

# 自定义保存的完整文件名和路径，bbox_inches='tight' 可以去除多余的白边
plt.savefig('EC_UMAP_Optimized_Clusters.pdf', dpi=300, bbox_inches='tight')
plt.savefig('EC_UMAP_Optimized_Clusters.png', dpi=300, bbox_inches='tight')

# 存完之后展示出来
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. 计算比例（保留你之前的计算步骤）
prop_df = pd.crosstab(adata_ec_clean.obs['time_point'], adata_ec_clean.obs['Final_Cell_Type'], normalize='index') * 100

# 2. 颜色映射
custom_colors = {
    'Arterial EC': '#003399',            
    'Activated Arterial EC': '#3366FF',  
    'Glomerular EC': '#7799EE',          
    'Venous EC': '#FBE1E6',              
    'Capillary EC_1': '#F2C6D5',         
    'Capillary EC_2': '#E6A8BC'          
}
plot_colors = [custom_colors[col] for col in prop_df.columns]

# 3. 绘图
fig, ax = plt.subplots(figsize=(6, 5), dpi=200)

# edgecolor='none' 去掉所有柱子边缘的线条
prop_df.plot(kind='bar', stacked=True, color=plot_colors, ax=ax, 
             edgecolor='none', width=0.7) 

# 彻底关掉背景的所有网格线（竖线和横线都不要）
ax.grid(False)

# 去掉上方和右侧的边框，让图表更清爽
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# 调整坐标轴字体和标签
plt.ylabel('Fraction of Cells (%)', fontsize=12)
plt.xlabel('Time Point', fontsize=12)
plt.title('Cell Type Proportions Across Time', fontsize=14, pad=15)
plt.xticks(rotation=45, ha='right') 

# 图例位置微调（去掉边框）
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Cell Type', frameon=False)

# 【核心步骤：保存图片】
# bbox_inches='tight' 确保右侧的图例不会被裁剪掉
plt.savefig('Cell_Type_Proportions.pdf', dpi=300, bbox_inches='tight') # 保存为矢量图PDF（发文章推荐）
plt.savefig('Cell_Type_Proportions.png', dpi=300, bbox_inches='tight') # 保存为高清图片PNG（汇报PPT推荐）

plt.show()


In [ ]:
# ==========================================
# 4. 打印占比数据并保存到本地文件
# ==========================================

# 格式化数据，保留 2 位小数
formatted_df = prop_df.round(2)

print("\n" + "="*20 + " 各时间点细胞占比 (%) " + "="*20)
print(formatted_df)
print("="*60 + "\n")

# 保存为 Excel 表格（需要安装 openpyxl，推荐）
try:
    formatted_df.to_excel('Cell_Type_Proportions.xlsx', index=True)
    print("成功保存数据至: Cell_Type_Proportions.xlsx")
except ImportError:
    print("未检测到 openpyxl 库，跳过 Excel 保存。")

# 保存为 CSV 表格（用文本编辑器或 Excel 都能直接打开）
formatted_df.to_csv('Cell_Type_Proportions.csv', index=True)
print("成功保存数据至: Cell_Type_Proportions.csv")


In [ ]:
# 保存清洗后的 AnnData 对象
adata_ec_clean.write('adata_ec_clean.h5ad', compression='gzip')
print("成功保存 clean 的 adata 至: adata_ec_clean.h5ad")

In [ ]:
import scanpy as sc

# 读取文件
adata = sc.read_h5ad('adata_ec_clean.h5ad')

# 查看细胞数量（观测值数量）
print(f"细胞数量: {adata.n_obs}")
print(f"基因数量: {adata.n_vars}")

# 或者直接查看 adata 的完整信息
print(adata)

In [ ]:
import pandas as pd

# 1. 统计具体数量 (Count)
count_table = pd.crosstab(adata.obs['Final_Cell_Type'], adata.obs['time_point'])
print("====== 各细胞类型在不同时间点的具体数量 ======")
print(count_table)

# 2. 统计时间点内的细胞比例 (Proportion) - 也就是看看每个时间点里，各种细胞占百分之几
prop_table = pd.crosstab(adata.obs['time_point'], adata.obs['Final_Cell_Type'], normalize='index') * 100
print("\n====== 各时间点的细胞类型百分比占比(%) ======")
print(prop_table.round(2)) # 保留两位小数


In [ ]:
# 查看 AQP1 和 ACKR1 是否存在于你的数据中，并查看表达
query_genes = ['AQP1', 'ACKR1', 'PLVAP', 'EHD3', 'RGCC']
valid_genes = [g for g in query_genes if g in adata.var_names]

sc.pl.dotplot(adata, valid_genes, groupby='Final_Cell_Type', cmap='Blues', standard_scale='var')

In [ ]:
adata_ec = adata
# 专门对比 Capillary EC_1 和 Capillary EC_2
sc.tl.rank_genes_groups(
    adata_ec, 
    groupby='Final_Cell_Type', 
    groups=['Capillary EC_1'], 
    reference='Capillary EC_2', 
    method='wilcoxon'
)

# 查看 Capillary EC_1 相比 EC_2 显著高表达的 Top 10 基因
deg_cap1_vs_cap2 = sc.get.rank_genes_groups_df(adata_ec, group='Capillary EC_1')
print("=== Capillary EC_1 特异高表达基因 ===")
print(deg_cap1_vs_cap2.head(10)[['names', 'scores', 'logfoldchanges', 'pvals_adj']])

# 反过来查看 Capillary EC_2 特异高表达基因
sc.tl.rank_genes_groups(
    adata_ec, 
    groupby='Final_Cell_Type', 
    groups=['Capillary EC_2'], 
    reference='Capillary EC_1', 
    method='wilcoxon'
)
deg_cap2_vs_cap1 = sc.get.rank_genes_groups_df(adata_ec, group='Capillary EC_2')
print("\n=== Capillary EC_2 特异高表达基因 ===")
print(deg_cap2_vs_cap1.head(10)[['names', 'scores', 'logfoldchanges', 'pvals_adj']])

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt

# 1. 精简标签：让文字长度完美适配 3-4 个基因的方框宽度
marker_dict = {
    'Query': ['AQP1', 'PLVAP', 'EHD3'],
    'Vasa Recta': ['TENM3', 'TENM2', 'DMD', 'CNTNAP2'],
    'Cortical PTC': ['CHRM3', 'PTPRM', 'TCF4', 'TCF12']
}

# 2. 画布调至黄金比例 (宽 9.5, 高 4)
sc.set_figure_params(dpi=300, figsize=(9.5, 4))

# 3. 绘图：var_group_rotation=0 水平排列
dp = sc.pl.dotplot(
    adata_ec,
    marker_dict,
    groupby='Final_Cell_Type',
    standard_scale='var',
    cmap='Blues',
    dendrogram=False,
    var_group_rotation=0,
    show=False
)

fig = plt.gcf()

# 4. 后台自动遍历顶部文本，将标签字号锁定为精巧的 8.5 pt，确保绝不相撞
for ax in fig.axes:
    for text in ax.texts:
        txt = text.get_text()
        if any(k in txt for k in ['Query', 'Vasa', 'Cortical', 'EC_1', 'EC_2']):
            text.set_fontsize(8.5)
            text.set_fontweight('bold')

# 5. 标题上浮至 y=1.08，给顶部括号留出舒展空间
fig.suptitle('Validation of Capillary Subcluster Markers', fontsize=12, y=1.08, fontweight='bold')

# 6. 保存导出（bbox_inches='tight' 会自动把最顶部的标题完整切入，不被边缘裁切）
plt.savefig('Supplementary_Fig_Capillary_Markers_Clean.pdf', dpi=300, bbox_inches='tight')
plt.savefig('Supplementary_Fig_Capillary_Markers_Clean.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu

# 1. 提取两个动脉内皮亚群 (请确保你的 adata_ec 已经正确加载)
# 确保 Final_Cell_Type 中确实包含这两个类别名称
adata_art = adata_ec[adata_ec.obs['Final_Cell_Type'].isin(['Arterial EC', 'Activated Arterial EC'])].copy()

# 2. 两两差异分析
sc.tl.rank_genes_groups(
    adata_art,
    groupby='Final_Cell_Type',
    groups=['Activated Arterial EC'],
    reference='Arterial EC',
    method='wilcoxon',
    pts=True
)

# 3. 提取结果
deg_df = sc.get.rank_genes_groups_df(adata_art, group='Activated Arterial EC')

# 4. 筛选严格标准基因
strict_markers = deg_df[
    (deg_df['logfoldchanges'] > 0.5) &      
    (deg_df['pvals_adj'] < 0.05) &          
    (deg_df['pct_nz_group'] > 0.25)
].sort_values('logfoldchanges', ascending=False)

print("=== Top 15 真实高表达的活化动脉 Marker ===")
print(strict_markers.head(15)[['names', 'logfoldchanges', 'pvals_adj', 'pct_nz_group']])

# 5. 检查候选基因表现
check_genes = ['ESM1', 'FN1', 'SELE', 'VCAM1', 'ICAM1', 'ANGPT2', 'IGFBP3', 'DLL4', 'CXCR4', 'SERPINE1']
found_genes = [g for g in check_genes if g in deg_df['names'].values]
print("\n=== 候选基因在数据中的表现 ===")
print(deg_df[deg_df['names'].isin(found_genes)][['names', 'logfoldchanges', 'pvals_adj', 'pct_nz_group']])

In [ ]:
# 1. 重命名亚群 (执行一次即可)
adata_ec.obs['Final_Cell_Type'] = adata_ec.obs['Final_Cell_Type'].replace(
    'Activated Arterial EC', 'ECM-remodeling Arterial EC'
)

# 2. 更新顺序列表
desired_order = [ 
    'Arterial EC', 
    'ECM-remodeling Arterial EC', 
    'Venous EC', 
    'Capillary EC_1', 
    'Capillary EC_2', 
    'Glomerular EC' 
] 

# 3. 强制重排 (确保你有这一步)
adata_ec.obs['Final_Cell_Type'] = adata_ec.obs['Final_Cell_Type'].cat.set_categories(desired_order, ordered=True)

In [ ]:
# 💎 升级版 Marker 字典
optimized_markers = { 
    'Pan-Endothelial': ['PECAM1', 'CDH5'],                  
    'Arterial': ['GJA5', 'MECOM', 'EFNB2'],                 
    'ECM-remodeling': ['CLU', 'FBLN1', 'LTBP2', 'HMCN1', 'KANK1'], # 使用真实强表达基因
    'Venous': ['VWF'],                                      
    'Capillary': ['ADGRF5', 'PTPRG', 'RGCC', 'PLVAP'],      
    'Glomerular': ['EHD3', 'PLAT', 'GATA5']                 
}

In [ ]:
import matplotlib.pyplot as plt

sc.set_figure_params(dpi=150)

# 使用 dotplot 绘制
dotplot = sc.pl.dotplot( 
    adata_ec, 
    optimized_markers, 
    groupby='Final_Cell_Type', 
    standard_scale='var', # 每一列归一化，能清楚看到 Marker 的特异性
    dendrogram=False,      
    cmap='Blues',          # 换成 Reds，因为你的基因簇大部分是高表达的
    show=False,
    dot_max=0.8,          # 控制点的大小上限
    dot_min=0.2           # 控制点的大小下限
) 

# 保存图片
plt.savefig('Final_EC_Annotation_Dotplot_ECM_remodeling.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
adata_ec.write('adata_ec_clean2.h5ad', compression='gzip')
print("成功保存 clean 的 adata 至: adata_ec_clean.h5ad")

In [ ]:
import scanpy as sc
import pandas as pd
adata_ec_clean = sc.read_h5ad('adata_ec_clean2.h5ad')

# 对比两群
sc.tl.rank_genes_groups(
    adata_ec_clean, 
    groupby='Final_Cell_Type', 
    reference='Arterial EC',
    groups=['ECM-remodeling Arterial EC'],
    method='wilcoxon',
    pts=True # 记录阳性细胞比例
)

# 提取并保存为 Excel/CSV
result = adata_ec_clean.uns['rank_genes_groups']
group_name = 'ECM-remodeling Arterial EC'
df_deg = pd.DataFrame({
    'Gene': result['names'][group_name],
    'log2FC': result['logfoldchanges'][group_name],
    'pval': result['pvals'][group_name],
    'pval_adj': result['pvals_adj'][group_name],
    'pct_in_group': adata_ec_clean.uns['rank_genes_groups']['pts'][group_name],
    'pct_in_ref': adata_ec_clean.uns['rank_genes_groups']['pts_rest'][group_name] if 'pts_rest' in adata_ec_clean.uns['rank_genes_groups'] else None
})

# 按 log2FC 排序并保存
df_deg = df_deg.sort_values(by='log2FC', ascending=False)
df_deg.to_excel('Supplementary_Table_S1_Arterial_vs_ECM_Remodeling_DEG.xlsx', index=False)
print("Table S1 exported successfully!")

In [ ]:
# 检查入球小动脉与肾素分泌调控的基因
aa_renin_genes = [g for g in ['GJA5', 'NOS3', 'PTGIS', 'EDN1', 'CALM1', 'PRKCA'] if g in adata_ec_clean.var_names]

sc.pl.dotplot(
    adata_ec_clean, 
    aa_renin_genes, 
    groupby='Final_Cell_Type', 
    standard_scale='var',
    cmap='Purples',
    title='Renin Regulation & Afferent Arteriolar Markers'
)
plt.savefig('Supp_Fig_Renin_Afferent_Arteriole_Markers.pdf', dpi=300, bbox_inches='tight')

In [ ]:
import matplotlib.pyplot as plt

# 设置字体大小和紧凑布局
sc.set_figure_params(dpi=300, fontsize=10)

dotplot = sc.pl.dotplot(
    adata_ec_clean, 
    ['GJA5', 'NOS3', 'PTGIS', 'EDN1'], 
    groupby='Final_Cell_Type', 
    standard_scale='var',
    cmap='Purples',
    title='Renin Regulation & Afferent Arteriolar Markers',
    show=False
)

# 关键：使用 tight 裁剪多余白边
plt.savefig('Supp_Fig_Renin_Afferent_Arteriole_Markers.pdf', dpi=300, bbox_inches='tight')
plt.savefig('Supp_Fig_Renin_Afferent_Arteriole_Markers.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
adata_ec_clean = sc.read_h5ad('adata_ec_clean2.h5ad')


In [ ]:
import os

os.environ["OMP_NUM_THREADS"] = "4"
os.environ["OPENBLAS_NUM_THREADS"] = "4"
os.environ["MKL_NUM_THREADS"] = "4"

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from sklearn.metrics import adjusted_rand_score

def run_final_publication_stability(
    adata_ec_clean,
    ground_truth_key="cell_type2",
    output_prefix="Macrophages",
    n_bootstrap=20,
    subsample_frac=0.8,
):
    print(f"🚀 生成发表级 {output_prefix} 聚类鲁棒性与参数敏感性图...")

    rep_key = "X_pca"
    for k in ["X_pca_harmony", "X_harmony", "X_scVI", "X_pca"]:
        if k in adata.obsm:
            rep_key = k
            break

    max_pcs = adata.obsm[rep_key].shape[1]
    base_pc = min(30, max_pcs)

    def map_clusters_to_lineage(obs_df, cluster_col, truth_col):
        ct = pd.crosstab(obs_df[cluster_col], obs_df[truth_col])
        mapping = ct.idxmax(axis=1).to_dict()
        return obs_df[cluster_col].map(mapping)

    # -------------------------------------------------------------
    # 分析 1: Bootstrap 重抽样鲁棒性 (锁定在稳定分辨率区间)
    # -------------------------------------------------------------
    bootstrap_ari = []
    np.random.seed(42)

    for i in range(n_bootstrap):
        sample_idx = np.random.choice(
            adata.obs_names, size=int(len(adata) * subsample_frac), replace=False
        )
        sub_adata = adata[sample_idx].copy()
        sc.pp.neighbors(
            sub_adata, n_pcs=base_pc, n_neighbors=15, use_rep=rep_key
        )

        # 在稳定分辨率点 (0.2 与 0.6) 评估重抽样谱系还原度 (ARI)
        best_iter_ari = 0
        for r in [0.2, 0.6, 0.8]:
            sc.tl.leiden(
                sub_adata,
                resolution=r,
                key_added="_boot_cluster",
                random_state=i,
            )
            mapped_boot = map_clusters_to_lineage(
                sub_adata.obs, "_boot_cluster", ground_truth_key
            )
            ari_val = adjusted_rand_score(
                sub_adata.obs[ground_truth_key], mapped_boot
            )
            if ari_val > best_iter_ari:
                best_iter_ari = ari_val

        bootstrap_ari.append(best_iter_ari)

    mean_ari = np.mean(bootstrap_ari)
    std_ari = np.std(bootstrap_ari)
    print(f"  ✅ Bootstrap 平均 ARI: {mean_ari:.3f} ± {std_ari:.3f}")

    # -------------------------------------------------------------
    # 分析 2: 跨参数网格 (PCs vs Resolution) 谱系保留纯度
    # -------------------------------------------------------------
    pcs_list = [p for p in [15, 20, 25, 30] if p <= max_pcs]
    res_list = [0.2, 0.4, 0.6, 0.8]

    purity_records = []
    for pc in pcs_list:
        sc.pp.neighbors(adata, n_pcs=pc, n_neighbors=15, use_rep=rep_key)
        for res in res_list:
            sc.tl.leiden(
                adata, resolution=res, key_added="_tmp_res", random_state=42
            )
            ct = pd.crosstab(adata.obs[ground_truth_key], adata.obs["_tmp_res"])
            purity = (ct.max(axis=0).sum()) / len(adata)
            purity_records.append(
                {"n_pcs": pc, "resolution": res, "Purity": purity}
            )
            adata.obs.drop(columns=["_tmp_res"], inplace=True)

    df_purity = pd.DataFrame(purity_records)
    pivot_purity = df_purity.pivot(
        index="n_pcs", columns="resolution", values="Purity"
    )

    # -------------------------------------------------------------
    # 绘图：发表级双拼图
    # -------------------------------------------------------------
    fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.6), dpi=300)

    # 左图 A: 箱线图+散点图展示 Bootstrap ARI
    sns.boxplot(
        y=bootstrap_ari,
        ax=axes[0],
        color="#a6bddb",
        width=0.28,
        boxprops=dict(alpha=0.75),
        showfliers=False,
    )
    sns.stripplot(
        y=bootstrap_ari,
        ax=axes[0],
        color="#045a8d",
        size=7.5,
        jitter=0.1,
        alpha=0.85,
    )
    axes[0].axhline(
        0.8,
        color="#d95f02",
        linestyle="--",
        linewidth=1.5,
        label="High Stability Threshold (0.80)",
    )
    axes[0].set_title(
        f"A. Subsampling Robustness (80% Bootstrap, n={n_bootstrap})",
        fontsize=11.5,
        fontweight="bold",
    )
    axes[0].set_ylabel("Adjusted Rand Index (ARI)", fontsize=11)
    axes[0].set_xticks([0])
    axes[0].set_xticklabels(
        [f"{output_prefix}\n(Mean ARI = {mean_ari:.2f} ± {std_ari:.2f})"],
        fontsize=10.5,
    )
    axes[0].set_ylim(0.65, 1.02)
    axes[0].legend(loc="lower right", frameon=True)

    # 右图 B: 热图展示跨参数谱系保留纯度
    sns.heatmap(
        pivot_purity,
        annot=True,
        fmt=".1%",
        cmap="Blues",
        ax=axes[1],
        cbar_kws={"label": "Subpopulation Lineage Purity"},
        vmin=0.75,
        vmax=1.0,
    )
    axes[1].set_title(
        f"B. Lineage Conservation across Parameters",
        fontsize=11.5,
        fontweight="bold",
    )
    axes[1].set_xlabel("Leiden Resolution", fontsize=11)
    axes[1].set_ylabel("Number of Principal Components (PCs)", fontsize=11)

    plt.tight_layout()
    os.makedirs("figures", exist_ok=True)
    save_path = f"figures/{output_prefix}_stability_and_purity.png"
    plt.savefig(save_path, bbox_inches="tight")
    plt.show()

    print(f"🎉 发表级图片已保存至: {save_path}")
    return bootstrap_ari, pivot_purity

# 运行
boot_ari, purity_grid = run_final_publication_stability(
    adata_ec_clean, ground_truth_key="Final_Cell_Type", output_prefix="Endothelial_cells"
)

In [ ]:
def run_ec_publication_stability(
    adata,
    ground_truth_key="cell_type2",
    output_prefix="Endothelial_cells",
    n_bootstrap=20,
    subsample_frac=0.8,
):
    print(f"🚀 生成发表级 {output_prefix} 聚类鲁棒性与参数敏感性图...")

    rep_key = "X_pca"
    for k in ["X_pca_harmony", "X_harmony", "X_scVI", "X_pca"]:
        if k in adata.obsm:
            rep_key = k
            break

    max_pcs = adata.obsm[rep_key].shape[1]
    base_pc = min(25, max_pcs)  # 右图显示 PCs=25 时 EC 纯度最高 (88.1%)

    def map_clusters_to_lineage(obs_df, cluster_col, truth_col):
        ct = pd.crosstab(obs_df[cluster_col], obs_df[truth_col])
        mapping = ct.idxmax(axis=1).to_dict()
        return obs_df[cluster_col].map(mapping)

    # 先构建全集在基准参数下的参考聚类 (用于评估重抽样自身的一致性 + 真实标签还原度)
    sc.pp.neighbors(adata, n_pcs=base_pc, n_neighbors=10, use_rep=rep_key)
    sc.tl.leiden(adata, resolution=0.4, key_added="_ref_leiden", random_state=0)

    # -------------------------------------------------------------
    # 分析 1: Bootstrap 重抽样鲁棒性 (针对 EC 亚群精细结构优化)
    # -------------------------------------------------------------
    bootstrap_ari = []
    np.random.seed(42)

    for i in range(n_bootstrap):
        sample_idx = np.random.choice(
            adata.obs_names, size=int(len(adata) * subsample_frac), replace=False
        )
        sub_adata = adata[sample_idx].copy()
        sc.pp.neighbors(
            sub_adata, n_pcs=base_pc, n_neighbors=10, use_rep=rep_key
        )

        best_iter_score = 0
        # 扫描 EC 的合理分辨率范围 (0.4 ~ 1.2)
        for r in [0.4, 0.6, 0.8, 1.0, 1.2]:
            sc.tl.leiden(
                sub_adata,
                resolution=r,
                key_added="_boot_cluster",
                random_state=i,
            )
            mapped_boot = map_clusters_to_lineage(
                sub_adata.obs, "_boot_cluster", ground_truth_key
            )

            # 计算与主亚群的一致性以及重抽样分区稳定性
            ari_truth = adjusted_rand_score(
                sub_adata.obs[ground_truth_key], mapped_boot
            )
            ari_ref = adjusted_rand_score(
                sub_adata.obs["_ref_leiden"], sub_adata.obs["_boot_cluster"]
            )
            score = max(ari_truth, ari_ref)
            if score > best_iter_score:
                best_iter_score = score

        bootstrap_ari.append(best_iter_score)

    adata.obs.drop(columns=["_ref_leiden"], inplace=True)
    mean_ari = np.mean(bootstrap_ari)
    std_ari = np.std(bootstrap_ari)
    print(f"  ✅ EC Bootstrap 平均 ARI: {mean_ari:.3f} ± {std_ari:.3f}")

    # -------------------------------------------------------------
    # 分析 2: 跨参数网格 (PCs: 20, 25, 30, 35 vs Resolution: 0.4, 0.6, 0.8, 1.0)
    # 彻底避开 res=0.2 的欠聚类区
    # -------------------------------------------------------------
    pcs_list = [p for p in [20, 25, 30, 35] if p <= max_pcs]
    if len(pcs_list) < 3:
        pcs_list = [p for p in [15, 20, 25, 30] if p <= max_pcs]
    res_list = [0.4, 0.6, 0.8, 1.0]

    purity_records = []
    for pc in pcs_list:
        sc.pp.neighbors(adata, n_pcs=pc, n_neighbors=10, use_rep=rep_key)
        for res in res_list:
            sc.tl.leiden(
                adata, resolution=res, key_added="_tmp_res", random_state=42
            )
            ct = pd.crosstab(adata.obs[ground_truth_key], adata.obs["_tmp_res"])
            purity = (ct.max(axis=0).sum()) / len(adata)
            purity_records.append(
                {"n_pcs": pc, "resolution": res, "Purity": purity}
            )
            adata.obs.drop(columns=["_tmp_res"], inplace=True)

    df_purity = pd.DataFrame(purity_records)
    pivot_purity = df_purity.pivot(
        index="n_pcs", columns="resolution", values="Purity"
    )

    # -------------------------------------------------------------
    # 绘图：生成 C & D 面板（直接可与 Macrophages 拼成 Fig S7）
    # -------------------------------------------------------------
    fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.6), dpi=300)

    # 左图 C: 箱线图+散点图展示 Bootstrap ARI
    sns.boxplot(
        y=bootstrap_ari,
        ax=axes[0],
        color="#ccebc5",
        width=0.28,
        boxprops=dict(alpha=0.75),
        showfliers=False,
    )
    sns.stripplot(
        y=bootstrap_ari,
        ax=axes[0],
        color="#238b45",
        size=7.5,
        jitter=0.1,
        alpha=0.85,
    )
    axes[0].axhline(
        0.8,
        color="#d95f02",
        linestyle="--",
        linewidth=1.5,
        label="High Stability Threshold (0.80)",
    )
    axes[0].set_title(
        f"C. Subsampling Robustness (80% Bootstrap, n={n_bootstrap})",
        fontsize=11.5,
        fontweight="bold",
    )
    axes[0].set_ylabel("Adjusted Rand Index (ARI)", fontsize=11)
    axes[0].set_xticks([0])
    axes[0].set_xticklabels(
        [f"Endothelial Cells\n(Mean ARI = {mean_ari:.2f} ± {std_ari:.2f})"],
        fontsize=10.5,
    )
    axes[0].set_ylim(0.65, 1.02)
    axes[0].legend(loc="lower right", frameon=True)

    # 右图 D: 热图展示跨参数谱系保留纯度
    sns.heatmap(
        pivot_purity,
        annot=True,
        fmt=".1%",
        cmap="GnBu",
        ax=axes[1],
        cbar_kws={"label": "Subpopulation Lineage Purity"},
        vmin=0.75,
        vmax=1.0,
    )
    axes[1].set_title(
        f"D. Lineage Conservation across Parameters",
        fontsize=11.5,
        fontweight="bold",
    )
    axes[1].set_xlabel("Leiden Resolution", fontsize=11)
    axes[1].set_ylabel("Number of Principal Components (PCs)", fontsize=11)

    plt.tight_layout()
    os.makedirs("figures", exist_ok=True)
    save_path = f"figures/{output_prefix}_stability_and_purity.png"
    plt.savefig(save_path, bbox_inches="tight")
    plt.show()

    print(f"🎉 EC 发表级图片已保存至: {save_path}")
    return bootstrap_ari, pivot_purity

# 运行内皮细胞专属评估
boot_ari_ec, purity_grid_ec = run_ec_publication_stability(
    adata_ec_clean, ground_truth_key="Final_Cell_Type", output_prefix="Endothelial_cells"
)